In [1]:
import pandas as pd
import unicodedata

input_path = "ekstraklasa_1989_2023.csv"
output_path = "ekstraklasa_1989_2023_normalized.csv"

df = pd.read_csv(input_path)

def strip_accents(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    return ''.join(c for c in text if not unicodedata.combining(c))

for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(strip_accents)

rename_map = {
    "data": "date",
    "godzina": "time",
    "dom": "home_team",
    "wyjazd": "away_team",
    "wynik": "score"
}

df = df.rename(columns=rename_map)

df[["home_goals", "away_goals"]] = df["score"].str.split(":", expand=True).astype(int)

df = df.drop(columns=["score", "Unnamed: 0", "time"])

df.to_csv(output_path, index=False)

In [2]:
#2. Wczytywanie danych

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss
#import matplotlib.pyplot as plt

df = pd.read_csv("ekstraklasa_1989_2023_normalized.csv", parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

df.head()

,date,home_team,away_team,home_goals,away_goals
0,1989-07-29,GKS Katowice,ŁKS Łodz,0,1
1,1989-07-29,Motor Lublin,Widzew Łodz,2,0
2,1989-07-29,Ruch Chorzow,Gornik Zabrze,2,0
3,1989-07-29,Legia Warszawa,FKS Stal Mielec,1,1
4,1989-07-29,Zawisza Bydgoszcz,Slask Wrocław,1,0


In [4]:
#3. Tworzenie 2 rekordów dla każdego meczu

In [4]:
def expand_matches_to_teams(df):
    rows = []

    for _, r in df.iterrows():
        # HOME TEAM
        rows.append({
            "date": r["date"],
            "team": r["home_team"],
            "opponent": r["away_team"],
            "is_home": 1,
            "goals_for": r["home_goals"],
            "goals_against": r["away_goals"],
        })

        # AWAY TEAM
        rows.append({
            "date": r["date"],
            "team": r["away_team"],
            "opponent": r["home_team"],
            "is_home": 0,
            "goals_for": r["away_goals"],
            "goals_against": r["home_goals"],
        })

    df_long = pd.DataFrame(rows)

    # target
    df_long["result"] = np.where(
        df_long["goals_for"] > df_long["goals_against"], 0,
        np.where(df_long["goals_for"] == df_long["goals_against"], 1, 2)
    )

    return df_long

df_long = expand_matches_to_teams(df)
df_long.head(6)

,date,team,opponent,is_home,goals_for,goals_against,result
0,1989-07-29,GKS Katowice,ŁKS Łodz,1,0,1,2
1,1989-07-29,ŁKS Łodz,GKS Katowice,0,1,0,0
2,1989-07-29,Motor Lublin,Widzew Łodz,1,2,0,0
3,1989-07-29,Widzew Łodz,Motor Lublin,0,0,2,2
4,1989-07-29,Ruch Chorzow,Gornik Zabrze,1,2,0,0
5,1989-07-29,Gornik Zabrze,Ruch Chorzow,0,0,2,2


In [5]:
#3. Rolling features

In [6]:
def add_team_rolling_features(df, window=5):
    df = df.sort_values("date")

    df["gf_roll"] = (
        df.groupby("team")["goals_for"]
        .rolling(window, min_periods=1)
        .mean()
        .shift(1)
        .reset_index(level=0, drop=True)
    )

    df["ga_roll"] = (
        df.groupby("team")["goals_against"]
        .rolling(window, min_periods=1)
        .mean()
        .shift(1)
        .reset_index(level=0, drop=True)
    )

    df["pts"] = np.where(
        df["goals_for"] > df["goals_against"], 3,
        np.where(df["goals_for"] == df["goals_against"], 1, 0)
    )

    df["pts_roll"] = (
        df.groupby("team")["pts"]
        .rolling(window, min_periods=1)
        .mean()
        .shift(1)
        .reset_index(level=0, drop=True)
    )

    return df

df = add_team_rolling_features(df_long)
df.head()

,date,team,opponent,is_home,goals_for,goals_against,result,gf_roll,ga_roll,pts,pts_roll
0,1989-07-29,GKS Katowice,ŁKS Łodz,1,0,1,2,2.0,3.0,0,0.8
15,1989-07-29,Jagiellonia Białystok,Zagłebie Sosnowiec,0,1,1,1,0.8,3.2,1,0.0
14,1989-07-29,Zagłebie Sosnowiec,Jagiellonia Białystok,1,1,1,1,1.4,1.0,1,2.0
13,1989-07-29,Lech Poznan,Zagłebie Lubin,0,0,2,2,1.4,1.2,0,1.6
12,1989-07-29,Zagłebie Lubin,Lech Poznan,1,2,0,0,0.4,2.0,3,0.2


In [7]:
#4. ELO

In [8]:
def add_elo_team_centric(df, k=30):
    elos = {}
    df["elo"] = np.nan

    for i, r in df.iterrows():
        team = r["team"]
        opp = r["opponent"]

        elos.setdefault(team, 1500)
        elos.setdefault(opp, 1500)

        df.at[i, "elo"] = elos[team]

        if r["result"] == 1:
            res = 1
        elif r["result"] == 0:
            res = 0.5
        else:
            res = 0

        exp = 1 / (1 + 10 ** ((elos[opp] - elos[team]) / 400))
        elos[team] += k * (res - exp)
        elos[opp] += k * ((1 - res) - (1 - exp))

    return df

df = add_elo_team_centric(df)
df.head()

,date,team,opponent,is_home,goals_for,goals_against,result,gf_roll,ga_roll,pts,pts_roll,elo
0,1989-07-29,GKS Katowice,ŁKS Łodz,1,0,1,2,2.0,3.0,0,0.8,1500.0
15,1989-07-29,Jagiellonia Białystok,Zagłebie Sosnowiec,0,1,1,1,0.8,3.2,1,0.0,1500.0
14,1989-07-29,Zagłebie Sosnowiec,Jagiellonia Białystok,1,1,1,1,1.4,1.0,1,2.0,1485.0
13,1989-07-29,Lech Poznan,Zagłebie Lubin,0,0,2,2,1.4,1.2,0,1.6,1500.0
12,1989-07-29,Zagłebie Lubin,Lech Poznan,1,2,0,0,0.4,2.0,3,0.2,1515.0


In [9]:
#5. Przygotowanie finalnych features

In [10]:
features = [
    "is_home",
    "gf_roll",
    "ga_roll",
    "pts_roll",
    "elo"
]

df_model = df.dropna(subset=features)
X = df_model[features].astype("float32")
y = df_model["result"].astype(int)
dates = df_model["date"]

In [11]:
#8. Split czasowy

In [12]:
split_date = dates.quantile(0.8)

X_train = X[dates <= split_date]
y_train = y[dates <= split_date]

X_test = X[dates > split_date]
y_test = y[dates > split_date]

len(X_train), len(X_test)

(14569, 3640)

In [13]:
#8a Brier funkcja

In [14]:
import numpy as np

def multiclass_brier(y_true, probs):
    y_onehot = np.zeros_like(probs)
    y_onehot[np.arange(len(y_true)), y_true] = 1
    return np.mean(np.sum((probs - y_onehot) ** 2, axis=1))

In [15]:
#9. Skalowanie + Logistic Regression (multinomial) + ewaluacja

In [16]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000
)

clf.fit(X_train_s, y_train)

proba_lr = clf.predict_proba(X_test_s)
pred_lr = clf.predict(X_test_s)
brier_lr = multiclass_brier(y_test, proba_lr)

print("Accuracy:", accuracy_score(y_test, pred_lr))
print("LogLoss:", log_loss(y_test, proba_lr))
print("LR Brier   :", brier_lr)

#sample = df_model[df_model.date > split_date].copy()
#sample["pred"] = preds
#sample[["date", "team", "opponent", "result", "pred"]].head(10)

Accuracy: 0.46593406593406594
LogLoss: 1.0340480050973342
LR Brier   : 0.6239071529390577


C:\Users\anton\anaconda3\envs\mlbet\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [17]:
#10. Random Forest

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)
brier_rf = multiclass_brier(y_test, proba_rf)

print("RF Accuracy:", accuracy_score(y_test, pred_rf))
print("RF LogLoss:", log_loss(y_test, proba_rf))
print("RF Brier   :", brier_rf)

RF Accuracy: 0.46263736263736266
RF LogLoss: 1.0392442886275481
RF Brier   : 0.6266992522394528


In [19]:
#11. XGBoost

In [20]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)
proba_xgb = xgb.predict_proba(X_test)
brier_xgb = multiclass_brier(y_test, proba_xgb)

print("XGB Accuracy:", accuracy_score(y_test, pred_xgb))
print("XGB LogLoss:", log_loss(y_test, proba_xgb))
print("XGB Brier   :", brier_xgb)

XGB Accuracy: 0.4623626373626374
XGB LogLoss: 1.0414671279638663
XGB Brier   : 0.6283671


In [21]:
#12. Porównywanie wyników

In [22]:
results = pd.DataFrame({
    "model": ["LogReg", "RandomForest", "XGBoost"],
    "accuracy": [
        accuracy_score(y_test, pred_lr),
        accuracy_score(y_test, pred_rf),
        accuracy_score(y_test, pred_xgb)
    ],
    "logloss": [
        log_loss(y_test, proba_lr),
        log_loss(y_test, proba_rf),
        log_loss(y_test, proba_xgb)
    ],
    "brier": [
        brier_lr,
        brier_rf,
        brier_xgb
    ]
})

results

,model,accuracy,logloss,brier
0,LogReg,0.465934,1.034048,0.623907
1,RandomForest,0.462637,1.039244,0.626699
2,XGBoost,0.462363,1.041467,0.628367


In [23]:
#13. Freature importance

In [24]:
pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

elo         0.453884
is_home     0.334805
pts_roll    0.086761
ga_roll     0.067103
gf_roll     0.057447
dtype: float64

In [25]:
pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=False)

is_home     0.645105
elo         0.146629
pts_roll    0.082595
ga_roll     0.064776
gf_roll     0.060895
dtype: float32

In [ ]:
#wnioski 
#ELO + HOME składają się na około 80% sygnału co wskazuje, że model uczy się prawdziwej struktury danych.
#Rolling features delikatnie poprawiają log-loss, z tego co wyczytałem, w przypadku klasyfikacji meczów piłki nożnej jest normalne.
#XGB działa na zasadzie drzewa decyzyjnego więc przewidywanym było że będzie faworyzował cechę is_home 
#Dla LG i RF można dodać interakcję is_home * elo -> powinno obniżyć log-loss i stabilizować predykcję 
#Można przeanalizować confusion matrix 0/1/2 